# FNO (masked-loss) vs. MUSIC — energy-density contours by η slice

Compares the TailMaskedLoss-retrained FNO model (`FNO4d` commit `dff91ba`,
see `PARTICLIZATION_INVESTIGATION.md`) against real MUSIC+iSS output,
overlaying CDF-based contours of the energy-density field at a chosen
spacetime-rapidity (η) bin, across 3 τ snapshots.

**"Truth" here is literally MUSIC's output** — these are the same ROOT
files (`AuAu_50ev_*_eta2_n33.root`) the model was trained/evaluated on,
generated by the real X-SCAPE/MUSIC/iSS pipeline, not a synthetic target.
This notebook runs the model fresh against held-out events from those
files — no live JetScape rerun needed.

All source settings live in **`eval_gcs_config_maskedloss.yaml`**. By
default it loads **only the 30-40% centrality file**: `train_torch.py`'s
train/verify split is a simple positional cut on the 4 training files
concatenated in centrality order, so for the full 200-event dataset the
held-out ~20% falls *entirely* inside 30-40%'s own block — the other three
centralities' events were all literally in the training set. To look at a
different centrality, edit `data_files` in the config to that file and set
`use_train_split: false` (its events are still real MUSIC output, just not
held-out — treat generalization claims from them accordingly), then re-run
from the top.

**Channels** (this model, n_features=5): `0=ε (energy density)`, `1=T`,
`2=vx`, `3=vy`, `4=vz`. Defaults to channel 0 below, the focus of this
notebook; `skinnycontour.plot_three_bins_contour`'s own per-channel title
labels (inherited from an older 4-channel model convention) don't match
this numbering past channel 0, so take the *dropdown* label as ground
truth if you switch channels, not the plot title.


In [ ]:
import io
import sys
import numpy as np
import matplotlib
matplotlib.use('Agg')          # non-interactive backend: no auto-display on figure
import matplotlib.pyplot as plt
plt.ioff()                     # belt-and-suspenders: keep interactive mode off
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output, Image

sys.path.insert(0, str(Path('.').resolve()))


In [ ]:
import eval_gcs_setup as _setup
try:
    _ctx = _setup.run_all(cfg_path='eval_gcs_config_maskedloss.yaml')
except Exception as _e:
    raise RuntimeError(f'Setup failed: {_e}') from _e

eval_cfg       = _ctx.eval_cfg
train_cfg      = _ctx.train_cfg
backend        = _ctx.backend
data_test      = _ctx.data_test
model_out_phys = _ctx.model_out_phys
y_true_phys    = _ctx.y_true_phys
eta_coords     = _ctx.eta_coords
tau_out_coords = _ctx.tau_out_coords
x_coords       = _ctx.x_coords
y_coords       = _ctx.y_coords
xy_extent      = _ctx.xy_extent
dtau           = _ctx.dtau
tau_min        = _ctx.tau_min
neta_eff       = _ctx.neta_eff
ntau_out       = _ctx.ntau_out

print(f'Loaded {data_test.shape[0]} event(s), {model_out_phys.shape[1]} channel(s), '
      f'{neta_eff} η bins, {ntau_out} τ steps.')


In [ ]:
_norm     = train_cfg.get('normalization', {})
norm_name = _norm.get('name', 'none') if isinstance(_norm, dict) else str(_norm)
_n_test   = data_test.shape[0]

_ws   = {'description_width': 'initial'}
_w400 = widgets.Layout(width='400px')

# --- channel labels for THIS model (n_features=5: ε, T, vx, vy, vz) --------
# Not the same numbering skinnycontour.plot_three_bins_contour's own title
# text assumes (an older 4-channel ε/vy/vx/vz convention) -- the dropdown
# label below is the correct one; the plot's title text is only right for
# channel 0.
_channel_options = [
    '0 — ε (energy density)',
    '1 — T (temperature)',
    '2 — vx',
    '3 — vy',
    '4 — vz',
][:model_out_phys.shape[1]]

cont_event_ui = widgets.IntSlider(
    min=0, max=_n_test - 1, value=0, step=1,
    description='Event:', layout=_w400, style=_ws,
)
cont_ieta_ui = widgets.IntSlider(
    min=0, max=neta_eff - 1, value=int(np.argmin(np.abs(eta_coords - 0.0))), step=1,
    description='η bin:', layout=_w400, style=_ws,
)
cont_channel_ui = widgets.Dropdown(
    options=_channel_options,
    value=_channel_options[0], description='Feature / channel:', style=_ws,
)
contour_iT_ui = widgets.Text(
    value='40,-1',
    description='',
    layout=widgets.Layout(width='120px'),
)
cdfrats_ui = widgets.Text(
    value='0.05,0.1,0.2,0.3,0.4,0.6',
    description='',
    layout=widgets.Layout(width='260px'),
)
# model_out_phys/y_true_phys are already denormalized to physical units
# (GeV/fm^3) by eval_gcs_setup's inference step -- rescale_energy=True
# would apply ANOTHER (e/tau)^(4/3) transform on top of that, which is only
# correct if the arrays were still in normalized (train-space) units. Off
# by default here on purpose, unlike eval_gcs_clean.ipynb's default (which
# assumed a different data flow). Leave off unless you know you want it.
rescale_energy_ui = widgets.Checkbox(
    value=False,
    description='Rescale energy: (e/τ)^(4/3)  [NOT needed for physical-unit data -- see comment above]',
    style=_ws,
)


In [ ]:
_controls = widgets.VBox([
    widgets.HBox([cont_event_ui, cont_ieta_ui, cont_channel_ui]),
    widgets.HBox([
        widgets.VBox([
            widgets.Label('2nd and 3rd τ-step indices (comma-sep; 1st panel always 0; -1=last non-zero):'),
            contour_iT_ui,
        ]),
        widgets.VBox([
            widgets.Label('CDF ratios for contour levels (comma-sep):'),
            cdfrats_ui,
        ]),
    ]),
    widgets.HBox([
        widgets.HTML(f'<b>Normalizer:</b> <code>{norm_name}</code>&nbsp;&nbsp;'),
        rescale_energy_ui,
    ]),
])

_run_btn = widgets.Button(
    description='Plot contours',
    button_style='primary',
    icon='play',
    layout=widgets.Layout(width='200px', margin='8px 0'),
)
_plot_out = widgets.Output()


def _show(fig):
    """Render fig to a PNG buffer and display as Image, then close the figure
    (matplotlib's Agg backend never auto-displays; this is the one render path)."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=fig.dpi, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.read()))


def _run_eval(b):
    from skinnycontour import plot_three_bins_contour

    with _plot_out:
        clear_output(wait=True)

        _event          = int(cont_event_ui.value)
        _channel        = int(cont_channel_ui.value.split()[0])
        _rescale_energy = bool(rescale_energy_ui.value)
        _raw_iT = [t.strip() for t in contour_iT_ui.value.split(',') if t.strip()]
        if len(_raw_iT) < 2:
            _raw_iT = (_raw_iT + ['-1', '-1'])[:2]
        elif len(_raw_iT) > 2:
            _raw_iT = _raw_iT[:2]
        _contour_iT = (0,) + tuple(int(t) for t in _raw_iT)
        _cdfrats    = [float(x.strip()) for x in cdfrats_ui.value.split(',') if x.strip()]
        _tau0       = tau_min + dtau
        _ieta       = int(cont_ieta_ui.value)
        _eta_phys   = eta_coords[_ieta]

        _fig, _axes = plt.subplots(1, 3, figsize=(16.0, 4.5))
        plot_three_bins_contour(
            model_out_phys, y_true_phys,
            iparam=_channel, event=_event,
            tau0=_tau0, taudelta=dtau, iT=_contour_iT,
            cdfrats=_cdfrats, ieta=_ieta, tag=f'η={_eta_phys:.2f}',
            rescale_energy=_rescale_energy,
            x_coords=x_coords, y_coords=y_coords,
            fig=_fig, axes=_axes,
        )
        _show(_fig)

        # Relative L2 error vs tau, all eta bins, for this channel -- quick
        # context on which eta/tau region the chosen snapshot sits in.
        _denom  = np.mean(y_true_phys[:, _channel] ** 2, axis=(0, 1, 2))
        _num    = np.mean((model_out_phys[:, _channel] - y_true_phys[:, _channel]) ** 2, axis=(0, 1, 2))
        _rel_l2 = np.sqrt(_num / np.maximum(_denom, 1e-30))
        _fig2, _ax2 = plt.subplots(figsize=(8, 4))
        for _ie in range(neta_eff):
            _ax2.plot(tau_out_coords, _rel_l2[_ie], label=f'η={eta_coords[_ie]:.2f}')
        _ax2.axvline(_tau0 + _contour_iT[0] * dtau, color='gray', ls=':', lw=1)
        _ax2.set_xlabel('τ (fm/c)')
        _ax2.set_ylabel('Relative L² error')
        _ax2.set_title(f'Relative L² error vs τ (all η bins, test set) — channel {_channel}')
        _ax2.legend(fontsize=7, ncol=2)
        _ax2.grid(True, alpha=0.3)
        _ax2.set_ylim(0, 2)
        _fig2.tight_layout()
        _show(_fig2)


_run_btn.on_click(_run_eval)

display(widgets.VBox([_controls, _run_btn, _plot_out]))
